In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv('../data/bank-additional-full.csv', sep=';')

## Removes Nulls & Drops Irrelevent Columns

In [3]:
# Where housing is missing, loan is missing
# Remove those rows
df = df[(df['housing'] != 'unknown') & (df['loan'] != 'unknown')]
# Drop jobs unknowns
df = df[df['job'] != 'unknown']
df = df[df['marital'] != 'unknown']
# Drop jobs unknowns
df = df[df['education'] != 'unknown']
cols_to_drop = ['default', 'duration', 'previous', 'pdays' ]
df.drop(columns=cols_to_drop, inplace=True)

Make plot to describe why scaling certain features (campaign)

In [4]:
# Encode target
y = (df['y'] == 'yes').astype(int)
X = df.drop(columns=['y'])

In [5]:
categorical_cols = ['job', 'marital', 'education', 'housing', 'loan',
                    'contact', 'month', 'day_of_week', 'poutcome']
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype=int)

## LogReg, use scaling

Keep campaign, scale campaign

In [6]:
X_train_log, X_test_log, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_log[numeric_cols] = scaler.fit_transform(X_train_log[numeric_cols])
X_test_log[numeric_cols] = scaler.transform(X_test_log[numeric_cols])

print(X_train_log.shape, X_test_log.shape, y_train.mean().round(3))

(30596, 43) (7649, 43) 0.111


## Random Forest, don't use scaling

In [7]:
# Evaluation for Random Forest
def evaluate_rf(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, y_pred, digits=3))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba):.3f}")
    print(f"PR-AUC:   {average_precision_score(y_test, y_proba):.3f}")

In [8]:
# Train/Test/Split Data
X_train_rf, X_test_rf, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model 1

In [9]:
# Train Model
rf1 = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
rf1.fit(X_train_rf, y_train)

# Results
evaluate_rf(rf1, X_test_rf, y_test)

              precision    recall  f1-score   support

           0      0.937     0.926     0.931      6797
           1      0.459     0.500     0.479       852

    accuracy                          0.879      7649
   macro avg      0.698     0.713     0.705      7649
weighted avg      0.883     0.879     0.881      7649

Confusion matrix:
 [[6295  502]
 [ 426  426]]
ROC-AUC:  0.787
PR-AUC:   0.456


# Model 2

In [10]:
# Train Model
rf2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf2.fit(X_train_rf, y_train)

# Results
evaluate_rf(rf2, X_test_rf, y_test)

              precision    recall  f1-score   support

           0      0.947     0.882     0.913      6797
           1      0.392     0.607     0.476       852

    accuracy                          0.851      7649
   macro avg      0.670     0.744     0.695      7649
weighted avg      0.885     0.851     0.865      7649

Confusion matrix:
 [[5996  801]
 [ 335  517]]
ROC-AUC:  0.801
PR-AUC:   0.471


In [11]:
(df == 'unknown').sum()

age               0
job               0
marital           0
education         0
housing           0
loan              0
contact           0
month             0
day_of_week       0
campaign          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

# Model 3 -- GridSearchCV Tuning

Using GridSearchCV, we can systematically search for the best hyperparameters for the random forest model rather than manually tuning or guessing. GridSearchCV will allow us to do this and test the models using cross-validation. 

In [25]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [6, 8, 12, None],
    'min_samples_leaf': [2, 10, 15, 20],
    'class_weight': ['balanced']
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',  # score based on ROC-AUC
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train_rf, y_train)

print("Best Parameters:", grid_search_rf.best_params_)
print(f"Best Cross-Validated ROC-AUC: {grid_search_rf.best_score_:.3f}")

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best Parameters: {'class_weight': 'balanced', 'max_depth': 12, 'min_samples_leaf': 10, 'n_estimators': 300}
Best Cross-Validated ROC-AUC: 0.798


## Evaluating Model 3 (Best GridSearch Estimator)


In [15]:
rf3 = grid_search_rf.best_estimator_

evaluate_rf(rf3, X_test_rf, y_test)

              precision    recall  f1-score   support

           0      0.947     0.882     0.914      6797
           1      0.393     0.608     0.477       852

    accuracy                          0.852      7649
   macro avg      0.670     0.745     0.695      7649
weighted avg      0.885     0.852     0.865      7649

Confusion matrix:
 [[5996  801]
 [ 334  518]]
ROC-AUC:  0.801
PR-AUC:   0.472


## Threshold Tuning on Model 3

Now we will tune the threshold down to 0.35. This will allow us to get a higher recall for the positive class (those who do invest in the bank after being contacted during the phone campaign). Although it will likely be at the expense of a lower precision, it is more valueable to reach the most possible potential investors rather than missing out on some. 

In [22]:
threshold = 0.35
y_pred_rf3_tuned = (y_prob_rf3 >= threshold).astype(int)

print(classification_report(y_test, y_pred_rf3_tuned, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf3_tuned))
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_rf3):.3f}")
print(f"PR-AUC:   {average_precision_score(y_test, y_prob_rf3):.3f}")

              precision    recall  f1-score   support

           0      0.955     0.713     0.817      6797
           1      0.242     0.731     0.364       852

    accuracy                          0.715      7649
   macro avg      0.599     0.722     0.590      7649
weighted avg      0.876     0.715     0.766      7649

Confusion matrix:
 [[4848 1949]
 [ 229  623]]
ROC-AUC:  0.801
PR-AUC:   0.472


## Final Random Forest Model Comparison

In [24]:
print("Model 1 — Default RF")
evaluate_rf(rf1, X_test_rf, y_test)

print("\nModel 2 — Tuned RF")
evaluate_rf(rf2, X_test_rf, y_test)

print("\nModel 3 — GridSearchCV RF (threshold 0.5)")
evaluate_rf(rf3, X_test_rf, y_test)

print("\nModel 3 — GridSearchCV RF (threshold 0.35)")
print(classification_report(y_test, y_pred_rf3_tuned, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf3_tuned))

Model 1 — Default RF
              precision    recall  f1-score   support

           0      0.937     0.926     0.931      6797
           1      0.459     0.500     0.479       852

    accuracy                          0.879      7649
   macro avg      0.698     0.713     0.705      7649
weighted avg      0.883     0.879     0.881      7649

Confusion matrix:
 [[6295  502]
 [ 426  426]]
ROC-AUC:  0.787
PR-AUC:   0.456

Model 2 — Tuned RF
              precision    recall  f1-score   support

           0      0.947     0.882     0.914      6797
           1      0.393     0.608     0.477       852

    accuracy                          0.852      7649
   macro avg      0.670     0.745     0.695      7649
weighted avg      0.885     0.852     0.865      7649

Confusion matrix:
 [[5996  801]
 [ 334  518]]
ROC-AUC:  0.801
PR-AUC:   0.472

Model 3 — GridSearchCV RF (threshold 0.5)
              precision    recall  f1-score   support

           0      0.947     0.882     0.914      67

GridSearchCV showed that we already had strong hyperparameters in Model 2, with the search returning nearly identical results. The threshold tuning proved to be effective in increasing recall, which correctly identified 105 additional investors that the default threshold would have missed. When we later compare this to our logistic regression, we can compare the precision and recall tradeoff to evaluate which model will be more valuable in our business context.